## 1.0 Calculo do ROI médio para esse projeto

In [ ]:
import pandas as pd

data = pd.read_csv("../data/HR-Employee-Attrition-balanced.csv")
data.head().T

In [ ]:
print("=" * 60)
# Numero total de trabalhadores
print(f"Numero total de trabalhadores: {len(data)}")

# Numero total de trabalhadores que saíram
total_left = data["Attrition"].value_counts()["Yes"]
print(f"Numero total de trabalhadores que saíram: {total_left}")

# Salário total anual dos trabalhadores
total_salary_total = (data["MonthlyIncome"].sum()) * 12
print(f"Salário total anual dos trabalhadores: {total_salary_total:.2f}")

# Salário total anual dos trabalhadores que sairam
total_salary_attrition = (data[data["Attrition"] == "Yes"]["MonthlyIncome"].sum()) * 12
print(f"Salário total anual dos trabalhadores que saíram: {total_salary_attrition:.2f}")

# Porcentagem dos salarios dos trabalhadores que sairam
percentual_salario = total_salary_attrition / total_salary_total
print(f"Porcentagem dos salarios dos trabalhadores que sairam: {percentual_salario:.2%}")

# Custo de turnover por funcionário - ver regra de negócio no PROBLEMA_NEGOCIO.MD

In [ ]:
# Adicionando uma nova coluna de valor de perda por saida
data_attrition = data.copy()

data_attrition_select = data_attrition.loc[
    data_attrition["Attrition"] == "Yes", :  # exemplo de colunas
]

data_attrition_select.head()

In [ ]:
# Criando uma regra para o joblevel
# Criando uma coluna de taxa de perda por nivel
# Se JobLevel == 1 retorne 0.4
# Se JobLevel == 2 retorne 1
# Se JobLevel == 3 retorne 1.5
# Se JobLevel == 4 ou > retorne 2

data_attrition_select["taxa_perda_por_nivel"] = data_attrition_select["JobLevel"].apply(
    lambda x: 0.4 if x == 1 else 1 if x == 2 else 1.5 if x == 3 else 2
)
data_attrition_select.head()

In [ ]:
# Calculando a perda financeira para cada funcionário que sair

data_attrition_select["salario_anual"] = data_attrition_select["MonthlyIncome"] * 12
data_attrition_select["perda_financeira"] = (
    data_attrition_select["salario_anual"] * data_attrition_select["taxa_perda_por_nivel"]
)
data_attrition_select.head()

In [ ]:
print("=" * 60)
# Numero total de trabalhadores
print(f"Numero total de trabalhadores: {len(data)}")

# Numero total de trabalhadores que saíram
total_left = data["Attrition"].value_counts()["Yes"]
print(f"Numero total de trabalhadores que saíram: {total_left}")

# Taxa de trabalhadores que sairam
taxa_saida = total_left / len(data)
print(f"Taxa de trabalhadores que sairam: {taxa_saida:.2%}")

# Salário total anual dos trabalhadores
total_salary_total = (data["MonthlyIncome"].sum()) * 12
print(f"Salário total anual dos trabalhadores: {total_salary_total:.2f}")

# Salário total anual dos trabalhadores que sairam
total_salary_attrition = (data[data["Attrition"] == "Yes"]["MonthlyIncome"].sum()) * 12
print(f"Salário total anual dos trabalhadores que saíram: {total_salary_attrition:.2f}")

# Porcentagem dos salarios dos trabalhadores que sairam
percentual_salario = total_salary_attrition / total_salary_total
print(f"Porcentagem dos salarios dos trabalhadores que sairam: {percentual_salario:.2%}")

# Custo de turnover por funcionário - ver regra de negócio no PROBLEMA_NEGOCIO.MD
custo_total_funcionario_perda = data_attrition_select["perda_financeira"].sum()
print(f"Custo total por turnover: {custo_total_funcionario_perda:.2f}")

# Custo médio de turnover por funcionário
custo_medio_turnover = data_attrition_select["perda_financeira"].mean()
print(f"Custo médio de turnover: {custo_medio_turnover:.2f}")

Recall = VP/(VP+FN)

# recall = VP / todos os positivos reais

In [ ]:
# Retorno

# recall = 0.3
# rh_retantion = 0.3 # taxa de eficiencia que o RH nao deixa o funcionário sair


def roi(recall, rh_retention):
    # Pessoas que o algoritmo aponta que irá sair e acerta
    true_positive = int(total_left * recall)

    # Funcionário retidos
    retention_employees = int(true_positive * rh_retention)

    # Custo médio de retenção

    custo_medio_retention = retention_employees * custo_medio_turnover

    # Investimentos

    ## Custo de infra
    custo_anual_infra = 130000

    ## Custo do data science para trabalhar
    custo_trabalhador = 200000

    ## Custo do RH para aumentar o salario de cada funcionário
    custo_rh = 5000

    total_anual_cost = custo_anual_infra + custo_trabalhador + (custo_rh * retention_employees)

    # ROI = RETORNO / INVESTIMENTO

    retorno_liquido = custo_medio_retention - total_anual_cost
    roi_anual = round((custo_medio_retention / total_anual_cost), 2)

    return {
        "cenario": "simulado",
        "recall": recall,
        "retantion": rh_retention,
        "func.retidos": retention_employees,
        "Economia liquida": retorno_liquido,
        "Roi anual": roi_anual,
    }


tabela_cenarios = [
    roi(recall=0.1, rh_retention=0.3),
    roi(recall=0.3, rh_retention=0.3),
    roi(recall=0.5, rh_retention=0.3),
    roi(recall=0.7, rh_retention=0.3),
    roi(recall=0.8, rh_retention=0.3),
]

df_potential = pd.DataFrame(tabela_cenarios)
df_potential = df_potential.style.format({"Economia liquida": "$ {:,.2f}"})
df_potential